In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
import pandas as pd
from trl import SFTTrainer
from transformers import TrainingArguments

In [ ]:
max_seq_length = 1024 
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name, max_seq_length = max_seq_length,
    dtype = None, load_in_4bit = True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r = 16, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32, lora_dropout = 0, bias = "none", use_gradient_checkpointing = "unsloth", random_state = 3407,
)

In [ ]:
prompt_template = """<|im_start|>system
Bạn là một chuyên gia ngôn ngữ học. Nhiệm vụ của bạn là phân tích tính châm biếm (sarcasm) trong bình luận dựa trên ngữ cảnh được cung cấp. BẮT BUỘC trả về định dạng JSON.<|im_end|>
<|im_start|>user
Input:
- Context: {}
- Comment: {}<|im_end|>
<|im_start|>assistant
```json
{{
  "results": [
    {{
      "comment": "{}",
      "sarcasm_label": "{}",
      "reasoning": "{}"
    }}
  ]
}}
```<|im_end|>"""

In [ ]:
df = pd.read_csv("gold_Qwen3.7_max.csv")

In [ ]:


def formatting_prompts_func(examples):
    texts = []
    for video, comment, label, reason in zip(examples['video_core_content'], examples['comment'], examples['sarcasm_label'], examples['reasoning']):
        clean_comment = str(comment).replace('"', '\\"')
        clean_reason = str(reason).replace('"', '\\"')
        text = prompt_template.format(video, comment, clean_comment, label, clean_reason)
        texts.append(text)
    return { "text" : texts }

In [ ]:
dataset = Dataset.from_pandas(df).map(formatting_prompts_func, batched = True)

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = dataset,
    dataset_text_field = "text", max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 8,  # Tối đa hóa cho RTX 5090
        gradient_accumulation_steps = 2,
        warmup_steps = 5, max_steps = -1, num_train_epochs = 3,
        learning_rate = 2e-4, fp16 = not torch.cuda.is_bf16_supported(), bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10, optim = "adamw_8bit", output_dir = "outputs",
    ),
)
trainer.train()
model.save_pretrained("qwen_7b_sarcasm_lora")
tokenizer.save_pretrained("qwen_7b_sarcasm_lora")
print("Train 7B XONG!")